# AML continual learning on Kaggle: CIFAR-10 -> CIFAR-100 -> Tiny ImageNet

Runs the AML supernet through three tasks of increasing class count and input size:

| task | classes | input | train images |
|---|---|---|---|
| `cifar10` | 10 | 32x32 | 50 000 |
| `cifar100` | 100 | 32x32 | 50 000 |
| `tinyimagenet` | 200 | 64x64 | 100 000 |

The supernet keeps **one shared classifier head** of width 200 (the max class
count over the sequence); each task is trained and read back on its own class
range `[0, n)`, and prototype rows are isolated per task, so two tasks that both
use slot 0 never interfere. Each task keeps its **native resolution** -- the
trunk is resolution-agnostic (fixed pooling at layers 5/11 + adaptive pooling
head), so downsampling Tiny ImageNet to 32x32 would only throw away detail.

### Datasets to add (right-hand *Add Input* panel)

Add a Kaggle dataset for each task; the exact slug does not matter, only that
the mounted folder contains the right child directory:

* CIFAR-10  -> a dataset containing `cifar-10-batches-py`
* CIFAR-100 -> a dataset containing `cifar-100-python`
* Tiny ImageNet -> a dataset containing `tiny-imagenet-200` (or any folder that
  holds `train/<wnid>/images`)

Cell 4 discovers them automatically, so you can add broadly and let the trainer
pick each task's root.

## 0. Environment

Expect a `2 x Tesla T4` accelerator. The trainer uses **one** GPU (it has no DDP
path), so the second T4 only helps if you run two experiments in parallel
notebooks. The fused CUDA kernels (`mgn.accel._C`) are **optional** and are not
built here: eager mode is the same arithmetic, just slower.

In [ ]:
import os, sys, glob, subprocess, torch

print("python ", sys.version.split()[0])
print("torch  ", torch.__version__)
print("cuda   ", torch.cuda.is_available(), "| devices", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print("  gpu", i, torch.cuda.get_device_name(i))
if not torch.cuda.is_available():
    print("!! no GPU: the run still works but Tiny ImageNet will be very slow")

# Pin one GPU so a 2xT4 box does not confuse anything.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")


## 1. Get the code

Clones the public repository into `/kaggle/working`, so every checkpoint the run
writes is part of this notebook's output. If your working copy is private, upload
it as a Kaggle Dataset instead and point `REPO_DIR` at the mount.

In [ ]:
REPO_URL = "https://github.com/AustraliaSilver/AML.git"
REPO_DIR = "/kaggle/working/AML"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())
print("cwd:", os.getcwd())


## 2. Find the data

Builds a `os.pathsep`-separated list of every mounted dataset folder. The trainer
resolves one root **per task** (the first that already holds that task's data),
so this list can safely contain folders for all three datasets at once. If
nothing is mounted, it falls back to `/kaggle/working/data` and `download=True`
fetches CIFAR-10/100 (needs *Internet* enabled); Tiny ImageNet has no downloader
and must be added as a dataset.

In [ ]:
# Two levels because a dataset may place the folder at its root or nested.
cands = sorted(glob.glob("/kaggle/input/*")) + sorted(glob.glob("/kaggle/input/*/*"))
roots = [d for d in cands if os.path.isdir(d)]
DATA_ROOT = os.pathsep.join(roots) if roots else "/kaggle/working/data"
print("data-root:")
for r in DATA_ROOT.split(os.pathsep):
    print("  ", r)


### Sanity check: can each task be located?

Cheap (no training). If a task prints `MISSING`, add that dataset or let it
download.

In [ ]:
from mgn.datasets import task_spec, resolve_task_root, dataset_present

for t in ["cifar10", "cifar100", "tinyimagenet"]:
    spec = task_spec(t)
    root = resolve_task_root(t, DATA_ROOT)
    ok = dataset_present(t, root)
    print(f"{t:14s} {spec.n_classes:3d} classes @ {spec.input_size}x{spec.input_size}  "
          f"{'OK  ' if ok else 'MISSING'}  {root}")


## 3. Smoke test (always run this first)

`--steps 30` caps each epoch at 30 batches, so this exercises the whole
multi-class / multi-resolution pipeline in a couple of minutes: the 200-wide head,
the per-task class range in the loss and in `evaluate`, the 64x64 Tiny ImageNet
path, and the three protocol matrices. If it prints a full `A[i,j]` matrix, the
full run will too.

In [ ]:
cmd = [
    sys.executable, "train_continual_aml.py",
    "--tasks", "cifar10,cifar100,tinyimagenet",
    "--epochs", "1,1,1", "--steps", "30",
    "--replay-per-class", "0",
    "--norm", "in", "--encoder-tail-act", "none",
    "--arch-lr", "0.003", "--depth-cost", "0.02",
    "--data-root", DATA_ROOT,
    "--workers", "2",
]
print(" ".join(cmd))
subprocess.run(cmd, check=True)


## 4. Full run

Same command with the `--steps` cap removed and real epoch counts.

**Measured cost** on this repo's RTX 3050 Laptop (`--max-dim 48`, batch 256,
`--workers 0`): CIFAR-10/100 ~0.9 s/step at 32x32, Tiny ImageNet ~12.5 s/step at
64x64. A T4 is roughly 2-3x faster, so the budget is on the order of:

| task | steps/epoch | 3050 (measured) | T4 (est.) |
|---|---|---|---|
| cifar10 | 195 | ~3 min | ~1-1.5 min |
| cifar100 | 195 | ~3 min | ~1-1.5 min |
| tinyimagenet | 390 @64x64 | ~80 min | ~30-40 min |

Tiny ImageNet dominates: `20,30,20` is roughly 10-14 h on a T4, at or past the
12 h session cap. Cheaper options, in order of how little they change the
experiment:

* fewer Tiny ImageNet epochs, e.g. `--epochs 20,30,8`;
* `--max-dim 32`: at 64x64 the ViT stage sees 16x16 = 256 tokens, and its
  attention is quadratic in the token count -- that is where the 64x64 time goes;
* `--workers 2` so JPEG decode overlaps compute.

**The trainer does not resume mid-sequence**: it saves
`aml_continual_after_<task>.pth` and `routing_state_after_<task>.pt` after each
task, but a fresh process always starts from task 0. Budget accordingly.

`--replay-per-class 0` is the zero-replay protocol the paper reports; Tiny Image
Net has no in-memory array and only supports 0 anyway.

In [ ]:
cmd = [
    sys.executable, "train_continual_aml.py",
    "--tasks", "cifar10,cifar100,tinyimagenet",
    "--epochs", "20,30,10",   # see the budget note above (20,30,20 needs ~10-14 h on a T4)
    "--replay-per-class", "0",
    "--norm", "in", "--encoder-tail-act", "none",
    "--arch-lr", "0.003", "--depth-cost", "0.02",
    "--data-root", DATA_ROOT,
    "--workers", "2",
]
print(" ".join(cmd))
subprocess.run(cmd, check=True)


## 5. Outputs

Written into the repo root (under `/kaggle/working`, so they persist as this
notebook's output):

* `aml_continual_after_cifar10.pth`, `..._after_cifar100.pth`, `..._after_tinyimagenet.pth`
  -- model weights after each task
* `routing_state_after_<task>.pt` -- router centroids / covariance and the
  per-task snapshots, saved after each task so a routing result can be
  re-examined offline

The three accuracy matrices printed at the end are:

* `A_oracle` -- Task-IL, task identity given (the zero-forgetting ceiling)
* `A_auto`   -- Domain-IL / Class-IL, task identity **predicted** by the router
* `A_live`   -- standard CL, current mutable weights (comparable to ER/EWC)

In [ ]:
for f in sorted(glob.glob(os.path.join(REPO_DIR, "*.pth")) + glob.glob(os.path.join(REPO_DIR, "*.pt"))):
    print(f"{os.path.getsize(f)/1e6:8.2f} MB  {os.path.basename(f)}")
